# Modelling a cash-settled ETO and handling instrument events

This Notebook demonstrates the concepts described in the following KBs:

1. Mastering a `ExchangeTradedOption` instrument with an underlying `MasteredInstrument`, establishing a position, and performing a valuation: https://support.lusid.com/docs/modelling-exchange-traded-options-in-lusid
2. Handling instrument events: https://support.lusid.com/docs/handling-lifecycle-events-for-exchangetradedoption-instruments

Note the following:

* The underlying must be an `Equity` or a `Future`; other underlyings do not support instrument events.
* Exercise is triggered by loading an **event instruction**, not `OptionExerciseCashEvent` itself.
* If `OptionExerciseCashEvent` is not triggered, LUSID emits `ExpiryEvent` automatically on the maturity date.

This Notebook creates two portfolios, one with an American Call option that gets exercised and another with a European Put that is not exercised and therefore expires. Both options have the same underlying BMW `Equity` instrument.

Option:
* Contract size: 100
* Contracts: 1
* Strike price: 80
* Start date: 15 Jan 2025
* Maturity date: 21 Mar 2025

Transaction:
* Units: 1000
* Price (of option): 3.13
* Fees: 0.17 * 1000 = 170
* Cost: (1000 * 3.13 * 100) + 170 = 313,170
* Transaction date: 13 Jan 2025
* Settlement date: 16 Jan 2025

**Note:** Currently, the transaction date must be *before* the instrument start date in order to generate instrument events.

In [1]:
import os
import pandas as pd
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.INFO)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

,apiVersion,buildVersion,excelVersion,links
0,v0,0.6.16015.0,0.5.3666,"{'relation': 'RequestLogs', 'href': 'https://j..."


In [2]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    instrument_events_api = api_factory.build(la.InstrumentEventsApi)
    instrument_event_type_api = api_factory.build(la.InstrumentEventTypesApi)
    corporate_action_sources_api = api_factory.build(la.CorporateActionSourcesApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

All APIs built correctly


## Create a scope and code for entities in the Notebook

Keep data segregated from other data in LUSID.

In [3]:
module_scope = "FBNTutorials"
module_code = "ETO-Cash"
print(f"'{module_scope}\\{module_code}' scope and code created.")

'FBNTutorials\ETO-CashTest4' scope and code created.


## Create a property type

To handle transaction fees.

In [4]:
def create_property_type(property_domain, property_scope, property_code, data_type):
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )

    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

In [5]:
fees_property = create_property_type("Transaction", "Fees", "TotalCapitalisedFees", "number")

INFO:root:Property type with the following key already exists: Transaction/Fees/TotalCapitalisedFees


## Create transaction types and sides

Required for both purchase transactions and for transactions automatically generated by instrument events.

All created in a custom transaction type scope, which must be registered with the portfolio in which transactions are loaded.

In [6]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e)

### Create sides

Must be created before transaction types. Includes recreating the built-in `Side1` and `Side2` in the custom transaction type scope.

In [7]:
# Recreate Side1 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TradeAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [8]:
# Recreate Side2 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:SettleCcy",
    currency = "Txn:SettlementCurrency",
    rate = "SettledToPortfolioRate",
    units = "Txn:TotalConsideration",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side2",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create transaction types

#### Create a `BuyETO` transaction type (to establish positions)

In [9]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "BuyETO",
            description = "Open the contract",
            transaction_class = "Options",
            transaction_roles = "LongLonger",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Side1",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashCommitment",
            side = "Side2",
            direction = -1
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration"
        ),
        # Total consideration is gross plus fees
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = f"Txn:GrossConsideration + Properties[{fees_property}]"
        )

    ]
)
try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "BuyETO",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `Expiry` transaction type (to handle `ExpiryEvent`)

Note LUSID only emits `ExpiryEvent` if an option is NOT exercised.

In [10]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "Expiry",
            description = "Transaction type for expiry event",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Set units to zero",
            movement_types = "StockMovement",
            side = "Side1",
            direction = -1
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "Expiry",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `CashSettledOptionExercise` transaction type (to handle `OptionExerciseCashEvent`)

Cash-settled options only. Triggered by event instruction. Impacts the `ExchangeTradedOption` holding, not the underlying.

In [11]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "CashSettledOptionExercise",
            description = "Transaction type for cash-settled option event",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Set units to zero",
            movement_types = "StockMovement",
            side = "Side1",
            direction = -1
        ),
        lm.TransactionTypeMovement(
            name = "Adjust cash balance",
            movement_types = "CashCommitment",
            side = "Side2",
            direction = 1
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "CashSettledOptionExercise",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


In [12]:
check_side("Side1", f"{module_scope}{module_code}")
check_side("Side2", f"{module_scope}{module_code}")
check_TT("BuyETO", f"{module_scope}{module_code}")
check_TT("Expiry", f"{module_scope}{module_code}")
check_TT("CashSettledOptionExercise", f"{module_scope}{module_code}")


Side1 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side1,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TradeAmount,0,None



Side2 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side2,Txn:SettleCcy,Txn:SettlementCurrency,SettledToPortfolioRate,Txn:TotalConsideration,Txn:TotalConsideration,0,None



BuyETO transaction type:


...

""



CashSettledOptionExercise transaction type:


,type,description,transaction_class,transaction_roles,is_default
0,CashSettledOptionExercise,Transaction type for cash-settled option event,Basic,AllRoles,False


,movement_types,side,direction,properties,mappings,name,movement_options,condition,settlement_mode
0,StockMovement,Side1,-1,{},[],Set units to zero,[],,Internal
1,CashCommitment,Side2,1,{},[],Adjust cash balance,[],,Internal


""


## Master instruments

The underlying `Equity` instrument is mastered separately first, and then included in the `ExchangeTradedOption` instrument as a `MasteredInstrument`. See https://support.lusid.com/docs/modelling-exchange-traded-options-in-lusid#mastering-an-instrument. 

In [13]:
def master_instrument(assetclass, id, currency, start, end, delivery, exercise, callorput, underlying_id):
    
    if assetclass == "equity":
        instrument_request = {
            id: lm.InstrumentDefinition(
                name = "BMW",
                identifiers = {
                    "ClientInternal": lm.InstrumentIdValue(value=id),
                    "Isin": lm.InstrumentIdValue(value = id)
                },
                definition = lm.Equity(instrument_type = "Equity", dom_ccy = currency),
            )
        }
    if assetclass == "eto":
        instrument_request = {
            id: lm.InstrumentDefinition(
                name = "BMW GR Equity OMON",
                identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id)},
                definition = lm.ExchangeTradedOption(
                    instrument_type = "ExchangeTradedOption", 
                    start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                    contractDetails=lm.ExchangeTradedOptionContractDetails(
                        domCcy=currency,
                        strike=80,
                        contractSize = 100,
                        description = "BMW GR Equity OMON",
                        # Required to load market prices against the underlying for valuation
                        underlyingCode = "MyIDForBMWQuotes",
                        country = "DE",
                        deliveryType = delivery,
                        exchangeCode = "EUREX",
                        exerciseDate = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                        exerciseType = exercise,
                        optionCode = "BMW",
                        optionType = callorput,
                        underlying = lm.MasteredInstrument(
                            instrumentType = "MasteredInstrument",
                            identifiers = {"Instrument/default/Isin": underlying_id}
                        )
                    ),
                    contracts = 1,
                    refSpotPrice = 0
                )
            )
        }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        # Return LUID from (only) instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id 
    except lu.ApiException as e:
        print(e)

In [14]:
luid_dict = {}
luid_dict["Equity_BMW"] = master_instrument("equity", "DE0005190003", "EUR", "", "", "", "", "", "")

In [15]:
luid_dict["ETO_BMW_2024-03-21_American_Cash_Call"] = master_instrument("eto", "ETO_BMW_2024-03-21_American_Cash_Call", "EUR", "2025-01-15", "2025-03-21", "Cash", "American", "Call", "DE0005190003")
luid_dict["ETO_BMW_2024-03-21_European_Cash_Put"] = master_instrument("eto", "ETO_BMW_2024-03-21_European_Cash_Put", "EUR", "2025-01-15", "2025-03-21", "Cash", "European", "Put", "DE0005190003")

for k, v in luid_dict.items():
    print(f"{k}: {v}")

Equity_BMW: LUID_00003H2C
ETO_BMW_2024-03-21_American_Cash_Call: LUID_00003H2D
ETO_BMW_2024-03-21_European_Cash_Put: LUID_00003H2E


In [16]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response, use_camel_case=True)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

,0,1,2
scope,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4
lusidInstrumentId,LUID_00003H2C,LUID_00003H2D,LUID_00003H2E
name,BMW,BMW GR Equity OMON,BMW GR Equity OMON
identifiers.ClientInternal,DE0005190003,ETO_BMW_2024-03-21_American_Cash_Call,ETO_BMW_2024-03-21_European_Cash_Put
identifiers.Isin,DE0005190003,NaN,NaN
identifiers.LusidInstrumentId,LUID_00003H2C,LUID_00003H2D,LUID_00003H2E
properties,[],[],[]
instrumentDefinition.instrumentType,Equity,ExchangeTradedOption,ExchangeTradedOption
instrumentDefinition.identifiers,{},NaN,NaN
instrumentDefinition.domCcy,EUR,NaN,NaN


## Create a recipe

Must be specified as a portfolio recipe to enable instrument events. Can also be used as a valuation recipe.

The recommended pricing model for an `ExchangeTradedOption` is `SimpleStatic`.

In [17]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-SimpleStatic",
    description = "A recipe to value an option",
    market = lm.MarketContext(
        market_rules = [
            # Look up FX spot rates in the LUSID quote store, if needed
            lm.MarketDataKeyRule(
                key = "Fx.CurrencyPair.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Rate",
                field = "mid",
                quote_interval = "1D.0D",
            ),
            # Rule for market prices for valuation (keyed by LusidInstrumentId)
            lm.MarketDataKeyRule(
                key = "Quote.LusidInstrumentId.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "1D.0D",
            ),
            # Rule for market prices for exercise (keyed by ISIN)
            lm.MarketDataKeyRule(
                key = "Quote.Isin.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "1D.0D",
            )
        ]
    ),
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="SimpleStatic",
                instrument_type="ExchangeTradedOption"
            )
        ]
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [18]:
# Confirm recipe upsert and show the many options that are automatically set to default values by LUSID.
config_recipe = recipe_api.list_configuration_recipes(filter=f"value.scope eq '{module_scope}' and value.code startswith '{module_code}'")
config_recipe_df = lusid_response_to_data_frame(config_recipe)
display(config_recipe_df.transpose())

,0
value.scope,FBNTutorials
value.code,ETO-CashTest4-SimpleStatic
value.market.market_rules.0.key,Fx.CurrencyPair.*
value.market.market_rules.0.supplier,Lusid
value.market.market_rules.0.data_scope,FBNTutorialsETO-CashTest4
value.market.market_rules.0.quote_type,Rate
value.market.market_rules.0.var_field,mid
value.market.market_rules.0.quote_interval,1D.0D
value.market.market_rules.0.price_source,
value.market.market_rules.0.source_system,Lusid


## Create corporate action source 

Required for now.

In [19]:
ca_source_definition = lm.CreateCorporateActionSourceRequest(
    scope=module_scope,
    code=module_code,
    display_name=f"{module_scope}/{module_code} CAS",
    instrument_scopes = [f"{module_scope}{module_code}"]
)

try:
    corporate_action_sources_api.create_corporate_action_source(
        create_corporate_action_source_request = ca_source_definition
    )
    print(f"{module_scope}/{module_code} CAS created")
except ApiException as e:
    print(e)

FBNTutorials/ETO-CashTest4 CAS created


## Set up two EUR transaction portfolios

The portfolio recipe is set to the recipe created above to enable instrument events, and the transaction type scope and corporate action source are registered.

In [20]:
def create_portfolio(name):
    portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name = f"{name} option portfolio",
        code = f"{module_code}-{name}",
        # Set the portfolio currency
        base_currency = "EUR",
        # Must be before first transaction recorded
        created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
        instrument_scopes = [f"{module_scope}{module_code}"],
        # Register transaction type scope
        transactionTypeScope=f"{module_scope}{module_code}",
        # Register portfolio recipe        
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-SimpleStatic"
            )
        ),
        # Register corporate action source
        corporate_action_source_id=lm.ResourceId(
            scope=module_scope,
            code=module_code
        )
    )

    try:
        portfolio_response=transaction_portfolios_api.create_portfolio(
            scope = module_scope,
            create_transaction_portfolio_request = portfolio_request
        )
        print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
    except ApiException as e:
        print(e)

In [21]:
ports = ["CashAmericanCall", "CashEuropeanPut"]
#ports = ["CashAmericanCall"]
for port in ports:
    create_portfolio(port)

Portfolio with display name 'CashAmericanCall option portfolio' created effective 2024-01-01 00:00:00+00:00
Portfolio with display name 'CashEuropeanPut option portfolio' created effective 2024-01-01 00:00:00+00:00


### Confirm portfolio details

In [22]:
def get_port_details(port):
    portfolio_response = transaction_portfolios_api.get_details(scope = module_scope, code = f"{module_code}-{port}")
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())
    
for port in ports:
    get_port_details(port)

,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,ETO-CashTest4-CashAmericanCall
base_currency,EUR
corporate_action_source_id.scope,FBNTutorials
corporate_action_source_id.code,ETO-CashTest4
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsETO-CashTest4
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsETO-CashTest4


,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,ETO-CashTest4-CashEuropeanPut
base_currency,EUR
corporate_action_source_id.scope,FBNTutorials
corporate_action_source_id.code,ETO-CashTest4
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsETO-CashTest4
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsETO-CashTest4


### Load transactions into portfolios

See https://support.lusid.com/docs/modelling-exchange-traded-options-in-lusid#booking-a-transaction-to-establish-a-position.

In [23]:
def create_transactions(port, txnid, tttype, luid, tradedate, settledate, quantity, price, ccy, fees):
    
    create_txn_request = {
        "number_one": lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=datetime.strptime(tradedate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            settlement_date=datetime.strptime(settledate, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            units=quantity,
            # This is the market price, used for the gross consideration calculation in the transaction type
            transaction_price=lm.TransactionPrice(
                price=price, type="Price"
            ),
            # Total consideration set to 0 to trigger the calculation in the transaction type
            total_consideration = lm.CurrencyAndAmount(
                currency = ccy,
                amount = 0
            ),
            properties={
                f"{fees_property}": lm.PerpetualProperty(
                    key = f"{fees_property}",
                    value = lm.PropertyValue(
                        metric_value = lm.MetricValue(
                            value = fees * quantity,
                        )
                    )
                )
            }
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}-{port}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e)

In [24]:
create_transactions("CashAmericanCall", "Txn01", "BuyETO", luid_dict["ETO_BMW_2024-03-21_American_Cash_Call"], "2025-01-13 00:00:00", "2025-01-16 00:00:00", 1000, 3.13, "EUR", 0.17)
create_transactions("CashEuropeanPut", "Txn01", "BuyETO", luid_dict["ETO_BMW_2024-03-21_European_Cash_Put"], "2025-01-13 00:00:00", "2025-01-16 00:00:00", 1000, 3.13, "EUR", 0.17)

Success
Success


### Confirm positions and audit output transactions

In [25]:
def get_portfolio_holdings(port, date):      
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))
    
    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            property_keys=["Instrument/default/Name"]
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        get_holdings_response_df.rename(columns = {
            "properties.Instrument/default/Name.value.label_value": "instrument"}, inplace = True)        
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties|sub_holding_keys').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
        return dict(get_holdings_response.values[0])['holding_id']
    except ApiException as e:
        print(e)

In [26]:
def get_output_transactions(port, start, end):
    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat()
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response, use_camel_case=True)
        display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e)

In [27]:
# Capture the holding ID of the option instrument in each portfolio, required to subsequently load event instructions
my_holdingid_dict = {}

for port in ports:
    print(f"\n{port}")
    my_holdingid_dict[port] = get_portfolio_holdings(port, "2025-01-16 23:59:59")  # Settlement date

for k, v in my_holdingid_dict.items():
    print(f"\n{k} holding ID: {v}")


CashAmericanCall


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsETO-CashTest4,LUID_00003H2D,BMW GR Equity OMON,P,"1,000.00","1,000.00","313,170.00",EUR,"313,170.00",EUR,EUR,Position,81212421,0.00,EUR,"313,170.00",EUR,"313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00
1,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212422,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00



CashEuropeanPut


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsETO-CashTest4,LUID_00003H2E,BMW GR Equity OMON,P,"1,000.00","1,000.00","313,170.00",EUR,"313,170.00",EUR,EUR,Position,81212423,0.00,EUR,"313,170.00",EUR,"313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00
1,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212424,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00



CashAmericanCall holding ID: 81212421

CashEuropeanPut holding ID: 81212423


## Valuation

See https://support.lusid.com/docs/modelling-exchange-traded-options-in-lusid#valuing-your-position.

Random valuation date: 30 Jan 2025

### Load market data

A market price is required both for the `ExchangeTradedOption` and the underlying `Equity` instrument for the valuation date.

Note the price for the underlying must be loaded into the Quote Store with a `LusidInstrumentId` of the `underlyingCode` specified in the `ExchangeTradedOption` instrument definition, in this case `MyIDForBMWQuotes`.

In [28]:
def load_quotes(id_type, id, price, date, ccy, scale):
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = id_type,
                    instrument_id = id,
                    quote_type = "Price",
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"{id} price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e)

In [29]:
# Option price
load_quotes("LusidInstrumentId", luid_dict["ETO_BMW_2024-03-21_American_Cash_Call"], 3.20, "2025-01-30 00:00:00", "EUR", 1)
load_quotes("LusidInstrumentId", luid_dict["ETO_BMW_2024-03-21_European_Cash_Put"], 3.20, "2025-01-30 00:00:00", "EUR", 1) 

# Underlying price
load_quotes("LusidInstrumentId", "MyIDForBMWQuotes", 82, "2025-01-30 00:00:00", "EUR", 1)

LUID_00003H2D price for 2025-01-30 00:00:00 successfully loaded into LUSID.
LUID_00003H2E price for 2025-01-30 00:00:00 successfully loaded into LUSID.
MyIDForBMWQuotes price for 2025-01-30 00:00:00 successfully loaded into LUSID.


In [30]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e)

list_quotes()

,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,LUID_00003H2D,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,3.20,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.117268+00:00,1.00
1,Lusid,LUID_00003H2E,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,3.20,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.400937+00:00,1.00
2,Lusid,MyIDForBMWQuotes,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,82.00,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.822627+00:00,1.00


### Perform valuation

In [31]:
def value_instruments(port, date, pnl_window):
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-SimpleStatic"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Value"),
            lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Value"),
            lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            lm.AggregateSpec(key="Valuation/Exposure", op="Value"),          
            lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"),
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Value", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}-{port}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),

    )

    try:
        # Get portfolio valuation
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Pricing model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Local Cost",
                "Valuation/CleanPV": "Local Clean PV",
                "Valuation/PV": "Local PV",
                "Valuation/Accrued": "Local Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Total/Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Total/Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
        val_response_df["Date"] = pd.to_datetime(val_response_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e)

In [32]:
for port in ports:
    print(f"\n{port}")
    value_instruments(port, "2025-01-30T17:00:00Z", "YTD") # Random valuation date


CashAmericanCall


,Category,Name,LUID,Pricing model,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Exposure,Notional,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,ExchangeTradedOption,BMW GR Equity OMON,LUID_00003H2D,SimpleStatic,2025-01-30,"1,000.00",3.20,"313,170.00","320,000.00","320,000.00",0.00,"8,200,000.00",100.00,"6,830.00","6,830.00",0.00,"6,830.00",0.00,[]
1,Cash,EUR,CCY_EUR,ConstantTimeValueOfMoney,2025-01-30,"-313,170.00",1.00,"-313,170.00","-313,170.00","-313,170.00",0.00,"-313,170.00",1.00,0.00,0.00,0.00,0.00,0.00,[]
2,Cash,EUR,CCY_EUR,None,2025-01-30,0.00,1.00,0.00,0.00,0.00,0.00,0.00,NaN,0.00,0.00,0.00,0.00,0.00,[]



CashEuropeanPut


,Category,Name,LUID,Pricing model,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Exposure,Notional,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,ExchangeTradedOption,BMW GR Equity OMON,LUID_00003H2E,SimpleStatic,2025-01-30,"1,000.00",3.20,"313,170.00","320,000.00","320,000.00",0.00,"8,200,000.00",100.00,"6,830.00","6,830.00",0.00,"6,830.00",0.00,[]
1,Cash,EUR,CCY_EUR,ConstantTimeValueOfMoney,2025-01-30,"-313,170.00",1.00,"-313,170.00","-313,170.00","-313,170.00",0.00,"-313,170.00",1.00,0.00,0.00,0.00,0.00,0.00,[]
2,Cash,EUR,CCY_EUR,None,2025-01-30,0.00,1.00,0.00,0.00,0.00,0.00,0.00,NaN,0.00,0.00,0.00,0.00,0.00,[]


## Instrument events

### Examine state before exercise

LUSID expects to emit `ExpiryEvent` automatically on the maturity date if no exercise occurs. 

`OptionExerciseCashEvent` is only emitted if triggered, but requires market data to generate a valid output transaction.

In [33]:
def query_instr_events(port):
    
    query_id_request = lm.QueryApplicableInstrumentEventsRequest(
        window_start = datetime.strptime("2015-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        window_end = datetime.strptime("2030-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        effective_at = datetime.strptime("2025-03-21", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        portfolio_entity_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{port}",
            )
        ],
        forecasting_recipe_id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-SimpleStatic"
        )
    )
    
    try:
        query_id_response = instrument_events_api.query_applicable_instrument_events(
            query_applicable_instrument_events_request = query_id_request,
            limit = 200
        )
        display(lusid_response_to_data_frame(query_id_response, use_camel_case=True).transpose())
    except ApiException as e:
        print(e)

In [34]:
for port in ports:
    print(f"\n{port}")
    query_instr_events(port)


CashAmericanCall


,0,1
portfolioId.scope,FBNTutorials,FBNTutorials
portfolioId.code,ETO-CashTest4-CashAmericanCall,ETO-CashTest4-CashAmericanCall
holdingId,81212421,81212421
lusidInstrumentId,LUID_00003H2D,LUID_00003H2D
instrumentScope,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4
instrumentType,ExchangeTradedOption,ExchangeTradedOption
instrumentEventType,OptionExerciseCashEvent,ExpiryEvent
instrumentEventId,LUID_00003H2D_OptionExerciseCashEvent_American,LUID_00003H2D_ExpiryEvent_20250321
generatedEvent.instrumentEventId,LUID_00003H2D_OptionExerciseCashEvent_American,LUID_00003H2D_ExpiryEvent_20250321
generatedEvent.instrumentIdentifiers.Instrument/default/ClientInternal,ETO_BMW_2024-03-21_American_Cash_Call,ETO_BMW_2024-03-21_American_Cash_Call



CashEuropeanPut


,0,1
portfolioId.scope,FBNTutorials,FBNTutorials
portfolioId.code,ETO-CashTest4-CashEuropeanPut,ETO-CashTest4-CashEuropeanPut
holdingId,81212423,81212423
lusidInstrumentId,LUID_00003H2E,LUID_00003H2E
instrumentScope,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4
instrumentType,ExchangeTradedOption,ExchangeTradedOption
instrumentEventType,OptionExerciseCashEvent,ExpiryEvent
instrumentEventId,LUID_00003H2E_OptionExerciseCashEvent_European,LUID_00003H2E_ExpiryEvent_20250321
generatedEvent.instrumentEventId,LUID_00003H2E_OptionExerciseCashEvent_European,LUID_00003H2E_ExpiryEvent_20250321
generatedEvent.instrumentIdentifiers.Instrument/default/ClientInternal,ETO_BMW_2024-03-21_European_Cash_Put,ETO_BMW_2024-03-21_European_Cash_Put


### Load an event instruction for the American call option to trigger `OptionExerciseCashEvent`

Requires a holding ID. See https://support.lusid.com/docs/handling-lifecycle-events-for-exchangetradedoption-instruments#exercising-a-cashsettled-option.

Random exercise date: 28 Feb 2025

In [35]:
def list_instructions_for_port(port):
    instructions_response = portfolios_api.list_instrument_event_instructions(scope = module_scope, code = f"{module_code}-{port}")
    instructions_response_df = lusid_response_to_data_frame(instructions_response)
    display(instructions_response_df)
    
for port in ports:
    print(f"\n{port}")
    list_instructions_for_port(port)


CashAmericanCall


""



CashEuropeanPut


""


In [36]:
# Show holding IDs
for k, v in my_holdingid_dict.items():
    print(f"{k}: {v}")

CashAmericanCall: 81212421
CashEuropeanPut: 81212423


In [37]:
def load_instruction_for_holding(port, instr, key, id, holding, exercisedate):
    
    if port == "CashAmericanCall":    
        instruction_request = {
            "load-instruction": lm.InstrumentEventInstructionRequest(
                instrumentEventInstructionId=f"{instr}_{key}_{id}",
                instrumentEventId=f"{instr}_{key}",
                instructionType="ElectForHolding",
                election_key="exercise",
                holdingId=holding,
                entitlementDateInstructed= to_date(exercisedate)
            )
        }
    # Can only exercise on expiry date, so entitlementDateInstructed not allowed...
    elif port == "CashEuropeanPut":
        instruction_request = {
            "load-instruction": lm.InstrumentEventInstructionRequest(
                instrumentEventInstructionId=f"{instr}_{key}_{id}",
                instrumentEventId=f"{instr}_{key}",
                instructionType="ElectForHolding",
                election_key="exercise",
                holdingId=holding
            )
        }
    else:
        instruction_request = {
            "load-instruction": lm.InstrumentEventInstructionRequest(
                instrumentEventInstructionId=f"{instr}_{key}_{id}",
                instrumentEventId=f"{instr}_{key}",
                instructionType="ElectForPortfolio",
                election_key="exercise",
            )
        }
    
    try:
        instruction_response = portfolios_api.upsert_instrument_event_instructions(
            scope = module_scope,
            code = f"{module_code}-{port}",
            request_body = instruction_request,
            success_mode="Partial"
        )
        print(instruction_response.failed) if instruction_response.failed else print("Success")
    except lu.ApiException as e:
        print(e)

In [38]:
load_instruction_for_holding("CashAmericanCall", luid_dict["ETO_BMW_2024-03-21_American_Cash_Call"], "OptionExerciseCashEvent_American", "instruction_1", my_holdingid_dict["CashAmericanCall"], "2025-02-28")
# load_instruction_for_holding("CashEuropeanPut", luid_dict["ETO_BMW_2024-03-21_European_Cash_Put"], "OptionExerciseCashEvent_European", "instruction_2", my_holdingid_dict["CashEuropeanPut"], "")

Success


In [39]:
for port in ports:
    print(f"\n{port}")
    list_instructions_for_port(port)


CashAmericanCall


,instrument_event_instruction_id,portfolio_id.scope,portfolio_id.code,instrument_event_id,instruction_type,election_key,holding_id,version.effective_from,version.as_at_date,version.as_at_created,version.user_id_created,version.request_id_created,version.reason_created,version.as_at_modified,version.user_id_modified,version.request_id_modified,version.reason_modified,version.as_at_version_number,version.entity_unique_id,href,ignore_cost_impact
0,LUID_00003H2D_OptionExerciseCashEvent_American...,FBNTutorials,ETO-CashTest4-CashAmericanCall,LUID_00003H2D_OptionExerciseCashEvent_American,ElectForHolding,exercise,81212421,0001-01-01 00:00:00+00:00,2026-06-16 09:09:43.121538+00:00,2026-06-16 09:09:43.121538+00:00,00u91lo2d7X42sdse2p7,2026061609-3f6aae1b180e4192b43a6db4eaede5c9,,2026-06-16 09:09:43.121538+00:00,00u91lo2d7X42sdse2p7,2026061609-3f6aae1b180e4192b43a6db4eaede5c9,,1,5dec17d3-9c56-4043-bbac-74a1c4fb8147,https://jamleed.lusid.com/api/api/portfolios/F...,False



CashEuropeanPut


""


### Load strike price for exercise date

`OptionExerciseCashEvent` requires a strike price for the underlying. See https://support.lusid.com/docs/handling-lifecycle-events-for-exchangetradedoption-instruments#loading-an-exercise-price-for-the-underlying-instrument-on-the-exercise-date

Note this price must be loaded into the Quote Store using an ISIN identifier for the underlying, not its LUID or the `underlyingCode` required for valuation.

In [40]:
load_quotes("Isin", "DE0005190003", 85, "2025-02-28 00:00:00", "EUR", 1)  # American call - must be higher than strike
# load_quotes("Isin", "DE0005190003", 75, "2025-03-21 00:00:00", "EUR", 1)  # European put - must be lower than strike

list_quotes()

DE0005190003 price for 2025-02-28 00:00:00 successfully loaded into LUSID.


,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,DE0005190003,Isin,Price,mid,12ed2e68-0195-446e-b9f2-e838fd2fd9e0,2025-02-28T00:00:00.0000000+00:00,85.00,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:44.743012+00:00,1.00
1,Lusid,LUID_00003H2D,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,3.20,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.117268+00:00,1.00
2,Lusid,LUID_00003H2E,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,3.20,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.400937+00:00,1.00
3,Lusid,MyIDForBMWQuotes,LusidInstrumentId,Price,mid,7ef5e21b-d065-4c18-971e-f7a63242bce5,2025-01-30T00:00:00.0000000+00:00,82.00,EUR,,,00u91lo2d7X42sdse2p7,2026-06-16 09:09:30.822627+00:00,1.00


### Examine final holdings

For the American Call option, `OptionExerciseCashEvent` is triggered and therefore emitted to exercise and record a profit (since the exercise price was above the strike price). `ExpiryEvent` is not emitted. See https://support.lusid.com/docs/handling-lifecycle-events-for-exchangetradedoption-instruments#examining-the-impact-of-optionexercisecashevent-on-the-portfolio.

For the European Put option, `OptionExerciseCashEvent` is **not** triggered and so `ExpiryEvent` is emitted instead to record a loss. See https://support.lusid.com/docs/handling-lifecycle-events-for-exchangetradedoption-instruments#expiring-an-unexercised-option.

In [41]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2025-01-16 23:59:59") # Settlement date
    get_portfolio_holdings(port, "2025-02-28 23:59:59") # Exercise date (for American option)
    get_portfolio_holdings(port, "2025-03-21 23:59:59") # Instrument expiry date


CashAmericanCall


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsETO-CashTest4,LUID_00003H2D,BMW GR Equity OMON,P,"1,000.00","1,000.00","313,170.00",EUR,"313,170.00",EUR,EUR,Position,81212421,0.00,EUR,"313,170.00",EUR,"313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00
1,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212422,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,default,CCY_EUR,EUR,B,"186,830.00","186,830.00","186,830.00",EUR,"186,830.00",EUR,EUR,Balance,81212422,0.00,EUR,"186,830.00",EUR,"186,830.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,default,CCY_EUR,EUR,B,"186,830.00","186,830.00","186,830.00",EUR,"186,830.00",EUR,EUR,Balance,81212422,0.00,EUR,"186,830.00",EUR,"186,830.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00



CashEuropeanPut


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsETO-CashTest4,LUID_00003H2E,BMW GR Equity OMON,P,"1,000.00","1,000.00","313,170.00",EUR,"313,170.00",EUR,EUR,Position,81212423,0.00,EUR,"313,170.00",EUR,"313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00
1,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212424,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsETO-CashTest4,LUID_00003H2E,BMW GR Equity OMON,P,"1,000.00","1,000.00","313,170.00",EUR,"313,170.00",EUR,EUR,Position,81212423,0.00,EUR,"313,170.00",EUR,"313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00
1,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212424,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00


,instrument_scope,instrument_uid,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,default,CCY_EUR,EUR,B,"-313,170.00","-313,170.00","-313,170.00",EUR,"-313,170.00",EUR,EUR,Balance,81212424,0.00,EUR,"-313,170.00",EUR,"-313,170.00",EUR,0.00,EUR,0.00,EUR,[],0.00,0.00


In [42]:
for port in ports:
    print(f"\n{port}")
    get_output_transactions(port, "2024-01-01", "2030-12-31")


CashAmericanCall


,0,1
transactionId,Txn01,LUID_00003H2D_OptionExerciseCashEvent_American...
type,BuyETO,CashSettledOptionExercise
description,Open the contract,Transaction type for cash-settled option event
instrumentIdentifiers.Instrument/default/LusidInstrumentId,LUID_00003H2D,LUID_00003H2D
instrumentScope,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4
instrumentUid,LUID_00003H2D,LUID_00003H2D
transactionDate,2025-01-13T00:00:00Z,2025-02-28T00:00:00Z
settlementDate,2025-01-16T00:00:00Z,2025-02-28T00:00:00Z
units,"1,000.00","1,000.00"
transactionAmount,"313,170.00","500,000.00"



CashEuropeanPut


,0,1
transactionId,Txn01,LUID_00003H2E_ExpiryEvent_20250321-81212423
type,BuyETO,Expiry
description,Open the contract,Transaction type for expiry event
instrumentIdentifiers.Instrument/default/LusidInstrumentId,LUID_00003H2E,LUID_00003H2E
instrumentScope,FBNTutorialsETO-CashTest4,FBNTutorialsETO-CashTest4
instrumentUid,LUID_00003H2E,LUID_00003H2E
transactionDate,2025-01-13T00:00:00Z,2025-03-21T00:00:00Z
settlementDate,2025-01-16T00:00:00Z,2025-03-21T00:00:00Z
units,"1,000.00","1,000.00"
transactionAmount,"313,170.00",0.00
